# Collections Analytics — Analysis Notebook

Reconstruct performance, test the 11% claim, document data-quality issues, and design the investment experiment.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
RAW = Path('../collections_assignment')


## 1. Inspect the supplied tables

In [ ]:
files = sorted(RAW.glob('*.csv'))
pd.DataFrame([{'table':p.name,'rows':len(pd.read_csv(p)),'columns':len(pd.read_csv(p,nrows=1).columns)} for p in files])


## 2. Clean payment events

`payment_id` is the transaction key. `payment_reference` is not assumed unique.

In [ ]:
payments = pd.read_csv(RAW/'payments.csv', parse_dates=['event_at'])
accounts = pd.read_csv(RAW/'accounts.csv')
valid_accounts = set(accounts.account_id)
payments['valid_success'] = payments.payment_status.eq('SUCCESS') & payments.account_id.isin(valid_accounts)
raw_recovery = payments.loc[payments.valid_success,'amount'].sum()
payments_clean = payments.sort_values(['payment_id','event_at']).drop_duplicates('payment_id',keep='first').copy()
payments_clean['valid_success'] = payments_clean.payment_status.eq('SUCCESS') & payments_clean.account_id.isin(valid_accounts)
clean_recovery = payments_clean.loc[payments_clean.valid_success,'amount'].sum()
pd.Series({'raw_success':raw_recovery,'clean_success':clean_recovery,'difference':raw_recovery-clean_recovery})


## 3. Reconstruct monthly performance

In [ ]:
targeting = pd.read_csv(RAW/'daily_targeting.csv', parse_dates=['target_date'])
targeting['month'] = targeting.target_date.dt.to_period('M').astype(str)
target_month = targeting.groupby(['account_id','month'],as_index=False).agg(target_days=('target_date','nunique'))
target_month = target_month.merge(accounts[['account_id','outstanding_amount','dpd','risk_segment','loan_type']],on='account_id',how='left')
m = target_month.groupby('month',as_index=False).agg(targeted_accounts=('account_id','nunique'),targeted_outstanding=('outstanding_amount','sum'))
payments_clean['month'] = payments_clean.event_at.dt.to_period('M').astype(str)
p = payments_clean[payments_clean.valid_success].groupby('month',as_index=False).agg(recovery=('amount','sum'))
m = m.merge(p,on='month',how='left').fillna({'recovery':0})
m['recovery_per_targeted_account'] = m.recovery/m.targeted_accounts
m['recovery_rate_on_targeted_outstanding'] = m.recovery/m.targeted_outstanding
m['mom_recovery_pct'] = m.recovery.pct_change()*100
m


In [ ]:
complete = m[m.month <= '2026-07']
plt.figure(figsize=(9,4))
plt.plot(complete.month, complete.recovery/1e7, marker='o')
plt.ylabel('Recovery (₹ Cr)')
plt.xlabel('Month')
plt.title('Monthly recovery after payment deduplication')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Test the 11% claim

A single 11% monthly increase is not the same as sustained improvement. Compare complete months and use denominator-aware metrics.

In [ ]:
complete[['month','recovery','mom_recovery_pct','recovery_per_targeted_account','recovery_rate_on_targeted_outstanding']]

## 5. Forensics

Check duplicate events, identity conflicts, timestamp chronology, denominator stability and attribution limitations.

In [ ]:
calls = pd.read_csv(RAW/'calls.csv', parse_dates=['event_at'])
status = pd.read_csv(RAW/'account_status_history.csv', parse_dates=['event_at','recorded_at'])
borrowers = pd.read_csv(RAW/'borrowers.csv')
print('Duplicate payment IDs:', payments.loc[payments.payment_id.duplicated(keep=False),'payment_id'].nunique())
print('Exact duplicate call rows:', calls.duplicated().sum())
print('Status chronology anomalies:', (status.recorded_at < status.event_at).sum())
print('Borrower IDs with >1 name:', borrowers.groupby('borrower_id').name.nunique().gt(1).sum())


## 6. Drivers

Descriptive cuts should cover portfolio mix, DPD, client/geography where reliable, campaign, channel, vendor, calling time, attempt frequency and borrower segment. The supplied data has no language field, so language cannot be analysed. Agent-level causal conclusions are avoided because agent master attributes conflict.

## 7. Counterfactual

Treatment = randomly assigned new targeting strategy. Control = existing strategy. Pre-register the outcome window and compare deduplicated recovery. Matching/regression/DID are alternatives only where the assignment structure supports them. State confounders, identification assumptions and limitations.

## 8. ₹10 Cr decision

Recommend better borrower targeting as a controlled, staged experiment. The supplied data does not support a defensible numerical causal lift, ROI or break-even estimate yet. Measure incremental recovery and verified costs before scaling.